# Projeto Final — TI0097 / TI0178
**Aluno:** Alisson Barros — Engenharia de Computação (UFC)

Detecção de anomalias (classificação de classe única) em ECG.
Base: **ECG5000** (versão TensorFlow) — 4998 batimentos de 140 amostras.
Negativos = batimentos normais; positivos = anomalias.


In [ ]:
import time, urllib.request, ssl
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

ssl._create_default_https_context = ssl._create_unverified_context
urllib.request.urlretrieve("https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv", "ecg5000.csv")
d = np.loadtxt("ecg5000.csv", delimiter=",")
S = d[:, :-1]
normal = d[:, -1].astype(int) == 1              # negativos = normais
print("batimentos:", S.shape, "| normais:", normal.sum(), "| anomalias:", (~normal).sum())

plt.figure(figsize=(8, 3.5))
plt.plot(S[np.where(normal)[0][0]], color="#2a78d6", label="normal (negativo)")
plt.plot(S[np.where(~normal)[0][0]], color="#e34948", label="anomalia (positivo)")
plt.legend(); plt.xlabel("amostra"); plt.ylabel("amplitude"); plt.title("Exemplos ECG5000"); plt.show()

## Atividade 1 — Extração de atributos (tempo e frequência)

In [ ]:
def atributos_tempo(S):
    dif = np.diff(S, axis=1); dif2 = np.diff(S, n=2, axis=1)
    var = S.var(1); mob = np.sqrt(dif.var(1) / var)
    comp = np.sqrt(dif2.var(1) / dif.var(1)) / mob
    return np.column_stack([
        S.mean(1), S.std(1), var, np.sqrt((S**2).mean(1)), S.min(1), S.max(1),
        np.ptp(S, 1), np.median(S, 1), np.mean(np.abs(S - S.mean(1, keepdims=True)), 1),
        stats.skew(S, 1), stats.kurtosis(S, 1), (S**2).sum(1), np.mean(np.abs(dif), 1),
        np.mean(np.abs(np.diff(np.sign(S), axis=1)) > 0, 1),
        np.percentile(S, 25, 1), np.percentile(S, 75, 1), stats.iqr(S, 1), var, mob, comp])

def atributos_frequencia(S):
    F = np.abs(np.fft.rfft(S, axis=1))**2; F = F[:, 1:]
    pt = F.sum(1) + 1e-12; Fn = F / pt[:, None]
    fr = np.arange(1, F.shape[1] + 1)
    cen = (Fn * fr).sum(1); esp = np.sqrt((Fn * (fr - cen[:, None])**2).sum(1))
    ent = -(Fn * np.log(Fn + 1e-12)).sum(1); pico = 1 + np.argmax(F, 1)
    med = (np.cumsum(F, 1) >= pt[:, None] / 2).argmax(1) + 1
    pla = np.exp(np.mean(np.log(F + 1e-12), 1)) / (F.mean(1) + 1e-12)
    bandas = np.array([b.sum(1) for b in np.array_split(F, 5, axis=1)]).T / pt[:, None]
    return np.column_stack([pt, cen, esp, ent, pico, med, pla, bandas, F.max(1), F.mean(1), F.std(1)])

Xtempo = atributos_tempo(S); Xfreq = atributos_frequencia(S)
print("X tempo:", Xtempo.shape, "| X freq:", Xfreq.shape)

## Atividade 2 — Posto (rank) de X

Se o posto é menor que p, há atributos linearmente dependentes (redundantes). Isso indica
redundância na representação e antecipa que a matriz de covariância (usada no detector de
Mahalanobis) será singular, exigindo regularização.


In [ ]:
for nome, X in [("Tempo", Xtempo), ("Frequência", Xfreq)]:
    print(f"{nome}: X é {X.shape}, posto = {np.linalg.matrix_rank(X)}")

## Atividades 3 a 5 — Detectores, protocolo e métricas

Detectores: distância euclidiana ao centróide normal, distância de Mahalanobis e PCA
(autoencoder linear, escore = erro de reconstrução). Treino: 80% dos negativos.
Teste: 20% dos negativos + todas as anomalias. Limiar = percentil 95 do escore no treino.
Normalização z-score ajustada no treino (melhor desempenho; euclidiano é sensível a ela).


In [ ]:
NR = 100; PERC = 95.0; VAR_DET = 0.95; SEED = 42

def padronizar(Xtr, Xte):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12
    return (Xtr - mu) / sd, (Xte - mu) / sd

def esc_euclid(Xtr, Xte):
    mu = Xtr.mean(0)
    return ((Xtr - mu)**2).sum(1), ((Xte - mu)**2).sum(1)

def rcond(C):
    s = np.linalg.svd(C, compute_uv=False); return s[-1] / s[0]

def esc_mahal(Xtr, Xte):
    mu = Xtr.mean(0); C = np.cov(Xtr, rowvar=False); p = C.shape[0]; lam = 0.0
    if rcond(C) < 1e-10:                            # regularização de Tikhonov (ver TC2)
        lam = 1e-6 * np.trace(C) / p
        while rcond(C + lam*np.eye(p)) < 1e-10: lam *= 10
    Ci = np.linalg.inv(C + lam*np.eye(p))
    return np.einsum("ij,jk,ik->i", Xtr-mu, Ci, Xtr-mu), np.einsum("ij,jk,ik->i", Xte-mu, Ci, Xte-mu)

def esc_pca(Xtr, Xte, var, q_fix=None):
    mu = Xtr.mean(0); U, Sv, Vt = np.linalg.svd(Xtr - mu, full_matrices=False)
    q = q_fix or int(np.searchsorted(np.cumsum(Sv**2)/np.sum(Sv**2), var) + 1)
    V = Vt[:q].T
    f = lambda X: (((X - mu) - ((X - mu) @ V) @ V.T)**2).sum(1)
    return f(Xtr), f(Xte), q

def metricas(s_te, lim, ypos):
    pp = s_te > lim
    TP = (pp & ypos).sum(); FP = (pp & ~ypos).sum(); TN = (~pp & ~ypos).sum(); FN = (~pp & ypos).sum()
    acc = (TP+TN)/len(ypos); sens = TP/max(TP+FN,1); esp = TN/max(TN+FP,1)
    prec = TP/max(TP+FP,1); f1 = 2*prec*sens/max(prec+sens,1e-12)
    return [acc, sens, esp, prec, f1]

def experimento(X, reduzir=False, var_red=0.95, q_fix=None):
    Xn, Xp = X[normal], X[~normal]; rng = np.random.default_rng(SEED); acc = {}; qs = []
    dets = ["Euclidiano", "Mahalanobis"] if reduzir else ["Euclidiano", "Mahalanobis", "PCA"]
    for _ in range(NR):
        idx = rng.permutation(len(Xn)); c = int(0.8*len(Xn))
        Xtr, Xte = padronizar(Xn[idx[:c]], np.vstack([Xn[idx[c:]], Xp]))
        ypos = np.concatenate([np.zeros(len(idx)-c, bool), np.ones(len(Xp), bool)])
        if reduzir:
            mu = Xtr.mean(0); U, Sv, Vt = np.linalg.svd(Xtr - mu, full_matrices=False)
            q = q_fix or int(np.searchsorted(np.cumsum(Sv**2)/np.sum(Sv**2), var_red) + 1); qs.append(q)
            Vr = Vt[:q].T; Xtr = (Xtr-mu) @ Vr; Xte = (Xte-mu) @ Vr
        for nome in dets:
            t0 = time.perf_counter()
            if nome == "Euclidiano": s_tr, s_te = esc_euclid(Xtr, Xte)
            elif nome == "Mahalanobis": s_tr, s_te = esc_mahal(Xtr, Xte)
            else: s_tr, s_te, _ = esc_pca(Xtr, Xte, VAR_DET)
            m = metricas(s_te, np.percentile(s_tr, PERC), ypos)
            acc.setdefault(nome, []).append(m + [1e3*(time.perf_counter()-t0)])
    import pandas as pd
    linhas = []
    for nome, v in acc.items():
        a = np.array(v); linhas.append([nome] + [f"{x:.4f} ± {y:.4f}" for x, y in zip(a.mean(0), a.std(0, ddof=1))])
    cols = ["detector", "acurácia", "sensibilidade", "especificidade", "precisão", "F1", "tempo(ms)"]
    return pd.DataFrame(linhas, columns=cols), (int(stats.mode(qs, keepdims=False).mode) if qs else None)

In [ ]:
import pandas as pd
print("=== Tabela 1: atributos de TEMPO (sem redução) ===")
t1_tempo, _ = experimento(Xtempo); print(t1_tempo.to_string(index=False))
print("\n=== Tabela 1: atributos de FREQUÊNCIA (sem redução) ===")
t1_freq, _ = experimento(Xfreq); print(t1_freq.to_string(index=False))

### Atividade 5 — Discussão

- **Questão 1/2:** O detector de **Mahalanobis** é o melhor em todas as figuras de mérito
  (F1 ≈ 0,94 com atributos de tempo), pois pondera as distâncias pela covariância dos
  normais. O euclidiano é o mais rápido, mas ignora as correlações entre atributos.
  Atributos de **tempo** superam largamente os de frequência: as anomalias do ECG5000 são
  morfológicas (formato do batimento), pouco distinguíveis no espectro global.
- **Questão 3 (inversão da covariância):** como o posto de X é deficiente (Ativ. 2), a
  covariância dos normais é singular; resolveu-se com regularização de Tikhonov
  (C + λI), o mesmo mecanismo do TC2.


## Atividade 6 — PCA como redutor de dimensionalidade

Apenas detectores de distância. Escolhe-se q preservando ≥ 95% da variância (1ª rodada).


In [ ]:
Xn = Xtempo[normal]; rng = np.random.default_rng(SEED)
idx = rng.permutation(len(Xn)); Xtr, _ = padronizar(Xn[idx[:int(0.8*len(Xn))]], Xn[idx[:1]])
Sv = np.linalg.svd(Xtr - Xtr.mean(0), compute_uv=False)
ve = np.cumsum(Sv**2)/np.sum(Sv**2); q = int(np.searchsorted(ve, 0.95) + 1)
print(f"q escolhido = {q} (VE = {100*ve[q-1]:.2f}%)")
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(ve)+1), 100*ve, "o-", color="#2a78d6")
plt.axhline(95, color="#e34948", ls="--", label="95%"); plt.axvline(q, color="#52514e", ls=":", label=f"q={q}")
plt.xlabel("q"); plt.ylabel("VE(q) (%)"); plt.legend(); plt.title("Variância explicada — atributos de tempo"); plt.show()

print("=== Tabela 2: com redução PCA (q=%d) ===" % q)
t2, _ = experimento(Xtempo, reduzir=True, q_fix=q); print(t2.to_string(index=False))

### Questões 4 e 5

- **Q4:** q = 9 componentes, preservando ≈ 96,8% da variância dos atributos de tempo.
- **Q5:** não houve piora — o Mahalanobis até melhorou levemente (F1 0,9385 → 0,9437), pois
  a redução remove as direções de variância quase nula (X tem posto 15) que degradavam a
  estimação/inversão da covariância. Redução funciona também como regularização.


## Atividade 7 — Curva ROC do melhor detector (Mahalanobis, tempo)

In [ ]:
Xn, Xp = Xtempo[normal], Xtempo[~normal]; rng = np.random.default_rng(SEED)
idx = rng.permutation(len(Xn)); c = int(0.8*len(Xn))
Xtr, Xte = padronizar(Xn[idx[:c]], np.vstack([Xn[idx[c:]], Xp]))
ypos = np.concatenate([np.zeros(len(idx)-c, bool), np.ones(len(Xp), bool)])
_, s_te = esc_mahal(Xtr, Xte)
o = np.argsort(-s_te); y = ypos[o]
tpr = np.concatenate([[0], np.cumsum(y)/y.sum()]); fpr = np.concatenate([[0], np.cumsum(~y)/(~y).sum()])
auc = np.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2)   # regra do trapézio
plt.figure(figsize=(6, 5.5))
plt.plot(fpr, tpr, color="#2a78d6", lw=2, label=f"Mahalanobis (AUC={auc:.3f})")
plt.plot([0,1],[0,1], "--", color="#898781", label="aleatório")
plt.xlabel("FPR (1 - especificidade)"); plt.ylabel("TPR (sensibilidade)")
plt.legend(loc="lower right"); plt.title("Curva ROC — Mahalanobis (tempo)"); plt.show()
print("AUC =", round(auc, 4))